# Event grain, session reconstruction, and behavioural features

**Research question:** Where does the cart-to-purchase opportunity remain after correcting event grain and session identity?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Raw session identifiers are not globally unique.
- Funnel conclusions should survive reasonable session-timeout choices.
- Price and category associations must not be described as causal effects.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: shopflow-product-analytics


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,prior_sessions,history,mart_session_funnel,user,Count sessions before first_eligible_at,strictly before eligible session,yes,none,zero,experiment covariate,NaN,low,none identified,point-in-time SQL predicate and unit tests
1,prior_cart_sessions,history,mart_session_funnel.reached_cart,user,Count prior sessions reaching cart,strictly before eligible session,yes,none,zero,audit,NaN,low,none identified,point-in-time SQL predicate
2,prior_purchase_sessions,history,mart_session_funnel.reached_purchase,user,Count prior sessions reaching purchase,strictly before eligible session,yes,none,zero,experiment covariate,NaN,low,may proxy relationship tenure,covariate balance SMD
3,prior_purchase_item_value,history,mart_session_funnel.purchase_item_value,user,Sum prior purchase item value,strictly before eligible session,yes,none,zero,CUPED covariate,NaN,low,spend may proxy customer value,CUPED precision comparison
4,prior_purchase_rate,history,prior purchase and session counts,user,prior_purchase_sessions / prior_sessions,strictly before eligible session,yes,none,zero when no history,experiment covariate,NaN,low,may proxy tenure,covariate balance SMD
5,returning_user,history,prior_sessions,user,1 when prior_sessions > 0,strictly before eligible session,yes,none,zero,heterogeneity slice,NaN,low,new-user status may proxy acquisition channel,pre-declared subgroup readout
6,price_quartile,product context,representative_price,session-product,Portfolio quantile bucket Q1-Q4,current viewed/carted product,yes,analysis population,unknown,association model,NaN,medium: conditions on carted products,price can proxy assortment/customer mix,adjusted GLM; non-causal interpretation
7,category_level_1,product context,category_code,session-product,Top-level category parsed from taxonomy,current product,yes,none,unknown,association model,NaN,low,category mix,cluster-robust GLM
8,activity_month,time control,session_started_at,session-product,YYYY-MM calendar month,current session,yes,none,unknown,association control,NaN,medium if used across future periods,seasonality,included as control only
9,converted_same_product,outcome,ordered full-funnel events,user/eligible session,Any ordered full-funnel product in eligible se...,after assignment,yes,none,false,target only,NaN,high if used as predictor,none,kept out of feature matrix


,feature_family,model_role,features
0,history,CUPED covariate,1
1,history,audit,1
2,history,experiment covariate,3
3,history,heterogeneity slice,1
4,outcome,target only,1
5,product context,association model,2
6,time control,association control,1


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,prior_sessions,strictly before eligible session,yes,none,low,point-in-time SQL predicate and unit tests
1,prior_cart_sessions,strictly before eligible session,yes,none,low,point-in-time SQL predicate
2,prior_purchase_sessions,strictly before eligible session,yes,none,low,covariate balance SMD
3,prior_purchase_item_value,strictly before eligible session,yes,none,low,CUPED precision comparison
4,prior_purchase_rate,strictly before eligible session,yes,none,low,covariate balance SMD
5,returning_user,strictly before eligible session,yes,none,low,pre-declared subgroup readout
6,price_quartile,current viewed/carted product,yes,analysis population,medium: conditions on carted products,adjusted GLM; non-causal interpretation
7,category_level_1,current product,yes,none,low,cluster-robust GLM
8,activity_month,current session,yes,none,medium if used across future periods,included as control only
9,converted_same_product,after assignment,yes,none,high if used as predictor,kept out of feature matrix


## Committed analytical evidence

In [4]:
evidence_files = ["reports/generated/cart_completion_model.csv", "reports/generated/opportunity_ranking.csv"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/generated/cart_completion_model.csv`

,term,coefficient,odds_ratio,ci_low,ci_high,p_value
0,Intercept,0.293828,1.341553e+00,8.028335e-01,2.241766e+00,2.620140e-01
1,"C(price_quartile, Treatment(reference=""Q1""))[T...",-0.109123,8.966201e-01,8.436070e-01,9.529645e-01,4.492474e-04
2,"C(price_quartile, Treatment(reference=""Q1""))[T...",-0.368247,6.919466e-01,6.463840e-01,7.407207e-01,3.108045e-26
3,"C(price_quartile, Treatment(reference=""Q1""))[T...",-0.472710,6.233107e-01,5.819133e-01,6.676531e-01,2.010586e-41
4,C(category_level_1)[T.apparel],-20.846385,8.841579e-10,1.169281e-10,6.685604e-09,1.060281e-90
5,C(category_level_1)[T.appliances],0.044375,1.045375e+00,6.268514e-01,1.743329e+00,8.649603e-01
6,C(category_level_1)[T.auto],0.021801,1.022041e+00,6.138936e-01,1.701545e+00,9.331932e-01
7,C(category_level_1)[T.computers],-0.174289,8.400537e-01,5.105546e-01,1.382203e+00,4.927196e-01
8,C(category_level_1)[T.construction],-0.147620,8.627590e-01,5.173450e-01,1.438794e+00,5.715760e-01
9,C(category_level_1)[T.country_yard],-0.093396,9.108330e-01,4.035673e-01,2.055708e+00,8.220762e-01


### `reports/generated/opportunity_ranking.csv`

,category_code,viewed_session_products,ordered_carts,ordered_purchases,median_cart_price,cart_completion_rate,gap_to_portfolio_benchmark,scenario_incremental_purchases,scenario_incremental_item_value
0,computers.components.videocards,72108,11713,4697,397.480,0.401007,0.075593,885.4,351935.07
1,computers.components.cpu,16533,2062,724,204.290,0.351115,0.125485,258.7,52859.87
2,computers.components.motherboard,18541,1958,769,73.810,0.392748,0.083852,164.2,12118.33
3,computers.peripherals.monitor,4997,388,146,202.950,0.376289,0.100311,38.9,7898.98
4,electronics.video.tv,17066,597,227,105.870,0.380235,0.096365,57.5,6090.72
5,computers.components.power_supply,6508,767,293,81.540,0.382008,0.094592,72.6,5915.91
6,computers.components.hdd,6414,739,291,73.240,0.393775,0.082825,61.2,4482.83
7,appliances.sewing_machine,4533,198,79,177.855,0.398990,0.077610,15.4,2733.06
8,construction.tools.drill,6717,341,140,71.400,0.410557,0.066043,22.5,1607.97
9,construction.tools.welding,2535,100,32,101.670,0.320000,0.156600,15.7,1592.15


## Decision

**Use reconstructed user-session grain and treat the opportunity ranking as a prioritisation hypothesis, not incremental revenue.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.